# TraffWise — YOLO26n Vehicle Detector Training (Kaggle)

Trains YOLO26n on the TraffWise vehicle dataset (classes: `bus, car, motorbike, truck`).

**Setup before running:**
1. Attach the vehicle dataset to this notebook (*Add Input*).
2. In the cell below, set `DATASET_ROOT` to the path of the folder that contains
   `images/` and `labels/` (copy the path from the sidebar file browser).
3. Enable a GPU accelerator (*Settings → Accelerator → GPU T4 x2 or P100*).
4. Run all cells. Checkpoints land in `/kaggle/working` — download `best.pt` when done.

**Session limits:** Kaggle GPU sessions cap at 12 h. If the run dies mid-training,
re-run the notebook — the train cell auto-resumes from `last.pt` if present in
`/kaggle/working`.

In [ ]:
%pip install -q -U ultralytics

import torch
import ultralytics

print("ultralytics", ultralytics.__version__)
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available())
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")

In [ ]:
# EDIT THIS: paste the path to the folder that contains images/ and labels/
# (find it in the sidebar: Add Input -> your dataset -> copy file path)
from pathlib import Path

DATASET_ROOT = Path("/kaggle/input/vehicle-detection-final/combination")

assert (DATASET_ROOT / "images/train").is_dir(), f"images/train not found under {DATASET_ROOT}"
assert (DATASET_ROOT / "labels/train").is_dir(), f"labels/train not found under {DATASET_ROOT}"

n_train = len(list((DATASET_ROOT / "images/train").glob("*.jpg")))
n_val = len(list((DATASET_ROOT / "images/val").glob("*.jpg")))
print(f"train images: {n_train}, val images: {n_val}")  # expect 17064 / 4280

In [ ]:
# Write data.yaml pointing at the read-only Kaggle input
data_yaml = f"""path: {DATASET_ROOT}
train: images/train
val: images/val

names:
  0: bus
  1: car
  2: motorbike
  3: truck
"""

with open("/kaggle/working/data.yaml", "w") as f:
    f.write(data_yaml)
print(data_yaml)

In [ ]:
import torch
from pathlib import Path
from ultralytics import YOLO

RUN_DIR = Path("/kaggle/working/runs/yolo26n")
last_ckpt = RUN_DIR / "weights/last.pt"
resume = last_ckpt.exists()

model = YOLO(str(last_ckpt) if resume else "yolo26n.pt")
print("Resuming from checkpoint" if resume else "Starting fresh from pretrained yolo26n.pt")

# Use every available GPU. Multi-GPU (DDP) needs a fixed batch size:
# autobatch (batch=-1) only works on a single GPU.
# nano @640 uses ~5 GB per 32 images, so 64 per 16 GB T4 fits comfortably.
n_gpu = torch.cuda.device_count()
device = list(range(n_gpu)) if n_gpu > 1 else 0
batch = 64 * n_gpu if n_gpu > 1 else -1
print(f"GPUs: {n_gpu}, device={device}, batch={batch}")

results = model.train(
    data="/kaggle/working/data.yaml",
    epochs=100,        # pretrained start converges early; patience stops it sooner anyway
    imgsz=640,         # keep 640 to stay comparable with the README's YOLO11 numbers
    batch=batch,
    device=device,
    workers=4,
    cache="disk",      # decode each image once; later epochs skip jpg decode (CPU bottleneck)
    patience=30,       # early stop if val mAP plateaus
    project="/kaggle/working/runs",
    name="yolo26n",
    exist_ok=True,
    resume=resume,
)

In [ ]:
# Final validation with the best checkpoint
best = YOLO(str(RUN_DIR / "weights/best.pt"))
metrics = best.val(data="/kaggle/working/data.yaml")
print(f"mAP50:    {metrics.box.map50:.4f}")
print(f"mAP50-95: {metrics.box.map:.4f}")
print("Per-class mAP50-95:", dict(zip(["bus", "car", "motorbike", "truck"], metrics.box.maps.round(4))))

In [ ]:
# Collect artifacts for download: best.pt + training log CSV
import shutil

shutil.copy(RUN_DIR / "weights/best.pt", "/kaggle/working/yolo26n_best.pt")
shutil.copy(RUN_DIR / "results.csv", "/kaggle/working/yolo26n_train_results.csv")
print("Saved: /kaggle/working/yolo26n_best.pt, yolo26n_train_results.csv")
print("Download from the notebook Output tab; results.csv follows the repo's evidence convention.")